In [ ]:
# import papermill as pm
import os
import subprocess

# import asyncio
# import platform

# if platform.system() == 'Windows':
#     asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# from spire.doc import *
# from spire.doc.common import *

In [ ]:
# Define the notebooks to run sequentially
notebooks_to_run = [
    "lor.snot22-1.ipynb",
    "lor.snot22-2.ipynb",
    "lor.snot22.ipynb",
    "lor.BBT-1.ipynb",
    "lor.BBT-2.ipynb",
    "lor.BBT-3.ipynb",
    "lor.breathing-1.ipynb",
    "lor.breathing-2.ipynb",
    "lor.breathing-3.ipynb",
    "lor.CBC-1.ipynb",
    "lor.CBC-2.ipynb",
    "lor.CBC-3.ipynb",
    "lor.CBC.ipynb",
    "lor.common.ipynb",
    "lor.common.morphology.ipynb",
    "lor.immune.ipynb",
    "lor.morphology-1-1.ipynb",
    "lor.morphology-1-1f.ipynb",
    "lor.morphology-1-2.ipynb",
    "lor.morphology-1-2f.ipynb",
    "lor.morphology-2.ipynb",
    "lor.morphology-2f.ipynb",
    "lor.morphology-3f.ipynb",
    "lor.morphology-4f.ipynb",
    # "morphological_analysis.ipynb",
    # "morphological_biopsy_analysis.ipynb",
    "time-event.ipynb"
]

In [ ]:
# Output path for executed notebooks (can be the same file using inplace)
input_dir = 'c:\\Analysis\\OTOLARING\\Nidelko\\jupyter'
output_dir = input_dir
# os.makedirs(output_dir, exist_ok=True)

In [ ]:
def execute_r_save_as_html(notebook_path, output_dir="reports"):
    """
    Executes a Jupyter Notebook (with an R kernel) and converts it to HTML.

    Args:
        notebook_path (str): The file path to the input .ipynb file.
        output_dir (str): The directory to save the output HTML file.
    """

    # Ensure the output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Define the output HTML path
    notebook_filename = os.path.basename(notebook_path)
    output_html_filename = os.path.splitext(notebook_filename)[0] + ".html"
    output_html_path = os.path.join(output_dir, output_html_filename)
    output_docx_filename = os.path.splitext(notebook_filename)[0] + ".docx"
    output_docx_path = os.path.join(output_dir, output_docx_filename)

    # Command to execute and convert the notebook
    # --execute runs the code cells, --to html specifies the format
    command = [
        "jupyter", "nbconvert",
        "--to", "html",
        "--no-input", "--no-prompt",
        "--execute",
        "--output", output_html_path,
        notebook_path
    ]

    try:
        # Run the command using subprocess
        subprocess.run(command, check=True,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(
            f" └Successfully executed '{notebook_path}' and saved as '{output_html_path}'")

        # print(f" └Create Word document {output_docx_path}")
        # # Create a Document object
        # document = Document()
        # # Load an HTML file
        # document.LoadFromFile(output_html_path, FileFormat.Html, XHTMLValidationType.none)

        # # Save the HTML file to a DOCX file
        # document.SaveToFile(output_docx_path, FileFormat.Docx2016)
        # # Or save the result document to a DOC file
        # # document.SaveToFile("HtmlToDoc.doc", FileFormat.Doc)

        # document.Close()

    except subprocess.CalledProcessError as e:
        print(f" └Error executing notebook: {e.stderr.decode()}")
    except FileNotFoundError:
        print("Error: jupyter nbconvert command not found. Ensure Jupyter is installed and in your PATH.")

In [ ]:
print(
    f"=========================================================================\n",
    f"Start processing ...\n",
    f"=========================================================================\n"
)

for notebook in notebooks_to_run:
    full_path = os.path.join(input_dir, notebook)
    print(f"Running {notebook}...")
    try:
        # pm.execute_notebook(b
        #     notebook,
        #     output_path,
        #     # You can pass parameters here if needed
        #     # parameters={"alpha": 0.5, "model_name": "mymodel"}
        # )
        # print(f"Successfully ran {notebook}")
        execute_r_save_as_html(notebook, input_dir)
        # print(input_dir)
    except Exception as e:
        print(f"Error running {notebook}: {e}")
        # break # Stop execution if an error occurs in one notebook

print(
    f"\n=========================================================================\n",
    f"End processing ...\n",
    f"=========================================================================\n"
)

In [ ]:
from ploomber import DAG
from ploomber.products import File
from ploomber.tasks import NotebookRunner
from ploomber.executors import Parallel, Serial

from pathlib import Path

dag = DAG(executor=Parallel(processes=6))

for notebook in notebooks_to_run:
    notebook_filename = os.path.basename(notebook)
    output_html_filename = os.path.splitext(notebook_filename)[0] + ".html"

    NotebookRunner(
        source=Path(notebook_filename),
        product=File(output_html_filename),
        # nb_product_key=['nb_ipynb', 'nb_html'],
        nbconvert_export_kwargs={'exclude_input': True},
        dag=dag,
        local_execution=True
        # name=str(i),
        # params=dict(
        #     a=10,
        #     b=i
        # )
    )

dag.build(force=True)